# Skin lesion classification: benign vs malignant

Three convolutional networks trained from scratch on the ISIC skin lesion
images (1,800 benign / 1,497 malignant, 224x224 RGB), and a post-hoc audit of
how they were evaluated.

The headline is not the accuracy. It is that the original evaluation had four
defects that make the three models **not directly comparable**, and finding
them is the more useful part of the exercise. Each is fixed below and flagged
with a `FIX` comment at the site.

| # | Defect | Consequence |
|---|---|---|
| 1 | `random_split` called with no generator | Each model saw a different validation set (supports 348/312, 354/306, 363/297) |
| 2 | The validation subset inherited `transform_train` | Accuracy was measured on randomly flipped, rotated and colour-jittered images; `transform_val` was dead code |
| 3 | `F.softmax` in `forward` **and** `nn.CrossEntropyLoss` | Softmax applied twice for two of the three models, flattening gradients. `DeeperCNN` does not do this, so the depth comparison is confounded |
| 4 | `roc_curve` fed `argmax` labels | The resulting "AUC" is balanced accuracy, not a threshold sweep |

The metrics quoted in the repository README come from the **original run**,
before these fixes. This notebook has not been re-run since; doing so needs a
GPU and the numbers should be expected to move.

In [ ]:
from pathlib import Path

# Resolve data relative to the repository root so the notebook runs whether
# Jupyter was started here or one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'skin-cancer'

# ImageFolder assigns labels in alphabetical order of the directory names.
CLASSES = ['benign', 'malignant']

SEED = 42

In [ ]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

## 1. The dataset

ISIC dermoscopic images, two classes. The EDA below confirms every image is already 224x224, so resizing is only needed to reach the 128x128 the networks expect.

In [ ]:
# The dataset is ~50 MB and is not committed. Needs a Kaggle API token in
# ~/.kaggle/kaggle.json. Skipped automatically once the data is present.
import subprocess, zipfile

if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    zip_path = DATA.parent / 'skin-cancer-isic-images.zip'
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'rm1000/skin-cancer-isic-images', '-p', str(DATA.parent)],
                   check=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
    zip_path.unlink()

print(sorted(p.name for p in DATA.iterdir()))

In [ ]:

import cv2
from PIL import Image

data_dir = DATA
classes = os.listdir(data_dir)

print("Classes and number of images:")
for class_name in classes:
    class_dir = os.path.join(data_dir, class_name)
    num_images = len(os.listdir(class_dir))
    print(f"{class_name}: {num_images}")

image_dimensions = []
image_sizes = []


for class_name in classes:
    class_dir = os.path.join(data_dir, class_name)
    sample_images = os.listdir(class_dir)[:10]
    for image_name in sample_images:
        image_path = os.path.join(class_dir, image_name)
        with Image.open(image_path) as img:
            image_dimensions.append(img.size)
            image_sizes.append(os.path.getsize(image_path) / 1024)


image_dimensions = np.array(image_dimensions)
image_sizes = np.array(image_sizes)

print("\nAverage image dimensions (width, height):", np.mean(image_dimensions, axis=0))
print("Average image size (KB):", np.mean(image_sizes))


def display_sample_images(data_dir, classes, num_images=5):
    plt.figure(figsize=(15, 15))
    for i, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        sample_images = os.listdir(class_dir)[:num_images]
        for j, image_name in enumerate(sample_images):
            image_path = os.path.join(class_dir, image_name)
            img = cv2.imread(image_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(len(classes), num_images, i * num_images + j + 1)
            plt.imshow(img)
            plt.axis('off')
            if j == 0:
                plt.ylabel(class_name, size=15)
    plt.show()


display_sample_images(data_dir, classes)


aspect_ratios = image_dimensions[:, 0] / image_dimensions[:, 1]


plt.figure(figsize=(10, 6))
plt.hist(aspect_ratios, bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of Image Aspect Ratios')
plt.xlabel('Aspect Ratio (Width / Height)')
plt.ylabel('Frequency')
plt.show()


## 2. Preprocessing and augmentation

Images are resized to 128x128 and normalised to [-1, 1]. Training adds horizontal flips, +/-20 degree rotations, scaling, colour jitter and random resized crops; validation applies resize and normalise only.

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


dataset_dir = DATA


image_size = 128
batch_size = 32


transform_train = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

transform_val = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
# FIX 1 and 2. The original was:
#
#     train_dataset = datasets.ImageFolder(root=dataset_dir, transform=transform_train)
#     val_dataset   = datasets.ImageFolder(root=dataset_dir, transform=transform_val)
#     train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [...])
#
# Two things went wrong there. The split had no generator, so every run - and
# so every model - got a different validation set. And the second line was
# immediately discarded by the third: both subsets came from the ImageFolder
# carrying `transform_train`, so validation accuracy was measured on randomly
# augmented images and `transform_val` was never used.

class TransformedSubset(torch.utils.data.Dataset):
    """A Subset that applies its own transform.

    random_split returns views onto one dataset, so both halves would
    otherwise share a single transform.
    """

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        image, label = self.subset[i]
        return self.transform(image), label


full_dataset = datasets.ImageFolder(root=dataset_dir)
assert full_dataset.classes == CLASSES, full_dataset.classes

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED))

train_dataset = TransformedSubset(train_subset, transform_train)
val_dataset = TransformedSubset(val_subset, transform_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"{len(train_dataset)} train / {len(val_dataset)} val")

## 3. Baseline architecture (`ArticleCNN`)

Four convolution/max-pool blocks (16 -> 32 -> 64 -> 128 channels) into three fully connected layers with dropout 0.5. This reproduces the architecture the assignment's reference paper specifies.

In [ ]:
class ArticleCNN(nn.Module):
    def __init__(self):
        super(ArticleCNN, self).__init__()


        self.conv2d_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.max_pooling2d_1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2d_2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.max_pooling2d_2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2d_3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.max_pooling2d_3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2d_4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.max_pooling2d_4 = nn.MaxPool2d(kernel_size=2, stride=2)



        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 64)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 2)

    def forward(self, x):

        x = F.relu(self.conv2d_1(x))
        x = self.max_pooling2d_1(x)

        x = F.relu(self.conv2d_2(x))
        x = self.max_pooling2d_2(x)

        x = F.relu(self.conv2d_3(x))
        x = self.max_pooling2d_3(x)

        x = F.relu(self.conv2d_4(x))
        x = self.max_pooling2d_4(x)


        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = F.relu(self.fc3(x))
        # FIX 3. Was F.softmax(...); nn.CrossEntropyLoss applies
        # log_softmax itself, so this squashed the gradients.
        x = self.output(x)
        return x

## 4. Training loop

Adam, cross-entropy, `ReduceLROnPlateau`, and early stopping on validation loss with patience 10 over a 50-epoch budget. The best checkpoint by validation loss is kept.

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=50, patience=10,
                checkpoint='best_model.pth'):
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    epochs_no_improve = 0

    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)

        train_loss /= len(train_loader.dataset)
        train_acc = correct_train / total_train
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)


        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)

                preds = torch.argmax(outputs, dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)

        val_loss /= len(val_loader.dataset)
        val_acc = correct_val / total_val
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
              f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")


        scheduler.step(val_loss)


        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1


        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    return history

## 5. Baseline results

Trained for 23 of 50 epochs before early stopping.

In [ ]:
model = ArticleCNN().to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, verbose=True)

history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=50, patience=10,
                      checkpoint='article_cnn.pth')

# Plot Accuracy
plt.figure(figsize=(10, 5))
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy per Epoch')
plt.legend()
plt.grid(True)
plt.show()

# Plot Loss
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss per Epoch')
plt.legend()
plt.grid(True)
plt.show()



In [ ]:
model.load_state_dict(torch.load('article_cnn.pth'))
model.eval()


print("\nClassification Report:")
y_true, y_pred, y_score = [], [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        # FIX 4. Keep P(malignant) too - a ROC curve needs a continuous
        # score. Feeding it argmax labels yields three points and an
        # 'AUC' that is just balanced accuracy.
        y_score.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())

print(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix Heatmap')
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## 6. Regularized variant (`ModifiedArticleCNN`)

Batch normalisation after every convolution, and global average pooling replacing the flatten. Global average pooling collapses each feature map to a single number, which removes the large `128*8*8 -> 512` matrix and most of the parameters with it.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ModifiedArticleCNN(nn.Module):
    def __init__(self):
        super(ModifiedArticleCNN, self).__init__()



        self.conv2d_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.max_pooling2d_1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2d_2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.max_pooling2d_2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2d_3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.max_pooling2d_3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2d_4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.max_pooling2d_4 = nn.MaxPool2d(kernel_size=2, stride=2)


        self.global_pooling = nn.AdaptiveAvgPool2d(1)


        self.fc1 = nn.Linear(128, 64)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(64, 32)
        self.dropout2 = nn.Dropout(0.4)
        self.output = nn.Linear(32, 2)

    def forward(self, x):

        x = F.relu(self.bn1(self.conv2d_1(x)))
        x = self.max_pooling2d_1(x)

        x = F.relu(self.bn2(self.conv2d_2(x)))
        x = self.max_pooling2d_2(x)

        x = F.relu(self.bn3(self.conv2d_3(x)))
        x = self.max_pooling2d_3(x)

        x = F.relu(self.bn4(self.conv2d_4(x)))
        x = self.max_pooling2d_4(x)


        x = self.global_pooling(x)
        x = torch.flatten(x, 1)


        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        # FIX 3. See ArticleCNN above.
        x = self.output(x)

        return x

In [ ]:
model = ModifiedArticleCNN().to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, verbose=True)

history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=50, patience=10,
                      checkpoint='modified_article_cnn.pth')

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy per Epoch')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss per Epoch')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.load_state_dict(torch.load('modified_article_cnn.pth'))
model.eval()


print("\nClassification Report:")
y_true, y_pred, y_score = [], [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        # FIX 4. Keep P(malignant) too - a ROC curve needs a continuous
        # score. Feeding it argmax labels yields three points and an
        # 'AUC' that is just balanced accuracy.
        y_score.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())

print(classification_report(y_true, y_pred, target_names=CLASSES))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix Heatmap')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## 7. Deeper variant (`DeeperCNN`)

Six convolutions in three double-conv blocks, no batch norm, no global pooling.

Worth noting before reading the numbers: this is the only one of the three that returns raw logits, so it is also the only one trained without the double-softmax of defect 3. Any comparison against the other two measures that difference as much as it measures depth.

In [ ]:

class DeeperCNN(nn.Module):
    def __init__(self):
        super(DeeperCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 16, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv4 = nn.Conv2d(32, 32, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv5 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv6 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)


        self.fc1 = nn.Linear(16384, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool1(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))
        x = self.pool3(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model = DeeperCNN()
print(model)




In [ ]:
model = DeeperCNN().to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, verbose=True)

history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=50, patience=10,
                      checkpoint='deeper_cnn.pth')

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy per Epoch')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss per Epoch')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.load_state_dict(torch.load('deeper_cnn.pth'))
model.eval()


print("\nClassification Report:")
y_true, y_pred, y_score = [], [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        # FIX 4. Keep P(malignant) too - a ROC curve needs a continuous
        # score. Feeding it argmax labels yields three points and an
        # 'AUC' that is just balanced accuracy.
        y_score.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())

print(classification_report(y_true, y_pred, target_names=CLASSES))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix Heatmap')
plt.show()